# Q1. Embedding a query

The embedder returns a vector of 384 numbers. What's the first value (v[0])?



In [1]:
from embedder import Embedder

2026-06-27 22:37:40.147697431 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
embedder = Embedder()

In [13]:
q1_query = "How does approximate nearest neighbor search work?"
q1_query_embedding = embedder.encode(q1_query)

In [14]:
q1_query_embedding.shape

(384,)

In [15]:
q1_query_embedding[0]

np.float64(-0.02058203437252893)

# Q2. Cosine similarity

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        sqliteseach_content = doc['content']

In [10]:
sqliteseach_content

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [17]:
sqliteseach_content_embedding = embedder.encode(sqliteseach_content)

In [18]:
q1_query_embedding.dot(sqliteseach_content_embedding)

np.float64(0.36107027225589694)

In [19]:
sqliteseach_content_embedding.dot(q1_query_embedding)

np.float64(0.36107027225589694)

# Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its filename)?

In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
chunk_content = []

for chunk in chunks:
    chunk_content.append(chunk['content'])

In [27]:
chunk_embeddings = embedder.encode_batch(chunk_content)

In [31]:
import numpy as np
X = np.array(chunk_embeddings)

In [32]:
scores = X.dot(q1_query_embedding)

In [33]:
scores

array([ 3.15187611e-01,  2.01479664e-01,  5.90559037e-02, -7.67661288e-02,
        1.18452466e-01, -1.41697012e-01, -2.81406495e-02, -4.65669117e-02,
       -2.06994543e-02, -6.07744147e-02,  2.13273769e-01,  8.87600958e-02,
       -1.97269268e-02,  3.11629985e-01,  5.51034635e-01,  2.04008152e-01,
        2.12515802e-01,  1.93649107e-01,  2.51961267e-01,  1.31078610e-01,
        2.59120607e-01,  7.63816369e-02,  9.59193203e-02,  9.81471228e-03,
       -3.59107168e-02,  1.01211406e-02,  1.10786940e-01, -9.90259915e-02,
       -3.71170645e-02,  7.59057333e-02, -3.35340234e-02,  8.86841484e-03,
        1.02636448e-01,  6.89615413e-02,  1.29408854e-01,  2.57709121e-01,
        3.23680576e-01,  1.06350076e-01,  5.61891208e-02,  2.34017441e-01,
        1.97954358e-01,  9.64296342e-02,  1.93709934e-01,  2.16719269e-01,
        3.48340450e-01,  5.10906541e-02,  2.05212833e-01,  1.05416182e-01,
       -3.25432660e-02,  4.94665347e-02,  2.38574873e-01, -3.44207304e-02,
        1.82165430e-01,  

In [34]:
np.argmax(scores)

np.int64(94)

In [35]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

# Q4. Vector search with minsearch

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

In [36]:
from minsearch import VectorSearch

In [39]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [40]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = vindex.search(query_vector, num_results=5)

In [42]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

# Q5. Text search vs vector search

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?



In [ ]:
from minsearch import Index


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [47]:
q5_query = "How do I store vectors in PostgreSQL?"

In [46]:
keyword_search_index = Index(
    text_fields=['content']
)

In [49]:
keyword_search_index.fit(chunks)

In [54]:
top5_keyword_search_results = keyword_search_index.search(q5_query)[:5]

In [57]:
q5_query_embedding = embedder.encode(q5_query)
top5_vector_search_results = vindex.search(q5_query_embedding, num_results=5)

In [63]:
top5_keyword_search_filenames = []
for result in top5_keyword_search_results:
    top5_keyword_search_filenames.append(result['filename'])

In [66]:
top5_keyword_search_filenames

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [64]:
top5_vector_search_filenames = []
for result in top5_vector_search_results:
    top5_vector_search_filenames.append(result['filename'])

In [67]:
top5_vector_search_filenames

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [70]:
filenames_only_vector_search = []
for vector_search_filename in top5_vector_search_filenames:
    if vector_search_filename not in top5_keyword_search_filenames:
        if vector_search_filename not in filenames_only_vector_search:
            filenames_only_vector_search.append(vector_search_filename)

In [71]:
filenames_only_vector_search

['02-vector-search/lessons/08-pgvector.md']

# Q6. Hybrid search

Which file is ranked first after RRF?

In [72]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [74]:
q6_query = "How do I give the model access to tools?"

In [75]:
text_results = keyword_search_index.search(q6_query)

In [76]:
q6_query_embedding = embedder.encode(q6_query)
vector_results = vindex.search(q6_query_embedding)

In [77]:
results = rrf([vector_results, text_results])

In [78]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca